# How an LLM Gets $\pi(y|x)$

This is the implementation-critical part of DPO.

We keep saying: $$ \pi_\theta(y|x) $$

but an LLM doesn't directly output "probability of this entire answer."

It predicts one token at a time.

---

## 1. Start with a simple example

**Prompt:** What is 2 + 2?

**Chosen response:** It is 4.

Tokenizer might turn it into:

- "It" → token 1
- "is" → token 2
- "4"  → token 3
- "."  → token 4

The LLM predicts:

$$ P(\text{It}|x) $$

then: $$ P(\text{is}|x,\text{It}) $$

then: $$ P(4|x,\text{It},\text{is}) $$

then: $$ P(.|x,\text{It},\text{is},4) $$

---

## 2. How do we get the probability of the whole answer?

Probability of a sequence is the product of the conditional probabilities:

$$ \boxed{ \pi(y|x) = \prod_{t=1}^{T} P(y_t|x,y_{<t}) } $$

So if the model gives:

- $P(\text{It})=0.8$
- $P(\text{is})=0.9$
- $P(4)=0.95$
- $P(.)=0.9$

then:

$$ \pi(y|x) = 0.8\times0.9\times0.95\times0.9 = 0.6156 $$

So the model assigns roughly **61.6% probability** to that exact sequence.

---

## 3. But we don't actually multiply these

Because sequences can contain hundreds/thousands of tokens.

Multiplying lots of numbers smaller than 1 causes **numerical underflow**.

Instead we use logs:

$$ \log\pi(y|x) = \sum_t \log P(y_t|x,y_{<t}) $$

For our example:

$$ \log\pi(y|x) = \log0.8+\log0.9+\log0.95+\log0.9 $$

**This is what we'll actually calculate in code.**

---

## 4. Where do these token probabilities come from?

The Transformer produces logits.

For example, at one position:

```
vocabulary
   ↓
logits
   ↓
softmax
   ↓
probabilities
```

Suppose: $$ z=[2.0,1.0,0.1] $$

Softmax: $$ P_i= \frac{e^{z_i}} {\sum_j e^{z_j}} $$

giving something like:

- token A → 0.659
- token B → 0.242
- token C → 0.099

The probability of the actual next token is the one we select from this distribution.

---

## 5. But there's a subtle implementation detail

Suppose the sequence is:

```
The cat sat
```

When predicting "cat", the model sees:

```
The
```

When predicting "sat", it sees:

```
The cat
```

So the model's logits are shifted relative to the target tokens.

Conceptually:

```
Input:     The    cat
Target:    cat    sat
```

**That's the same causal-language-model setup used during normal cross-entropy training.**

For DPO, we use that mechanism to calculate the likelihood of the already-given response, rather than generating it.

---

## 6. This is important: DPO does NOT need to generate the answer

This is one of the biggest differences from PPO RLHF.

We already have:

- prompt
- chosen response
- rejected response

DPO simply asks the model:

> "How probable is this existing response?"

So:

```
chosen ─────→ πθ → log P(chosen)
rejected ───→ πθ → log P(rejected)
```

and the same thing through the frozen reference model:

```
chosen ─────→ πref → log P(chosen)
rejected ───→ πref → log P(rejected)
```

**No sampling is required.**

---

## 7. We only score the response, not the prompt

Suppose:

- **Prompt:** Explain gravity.
- **Response:** Gravity is an attractive force...

We don't want:

$$ \log P(\text{Explain}) + \log P(\text{gravity}) +\cdots $$

included in the DPO score.

We want:

$$ \boxed{ \log P(\text{response}|\text{prompt}) } $$

So during implementation we create a **loss mask**:

```
Prompt tokens:    0 0 0 0 0
Response tokens:  1 1 1 1 1
```

Only tokens marked 1 contribute.

---

## 8. Why does this matter?

Because otherwise we'd be measuring:

> "How likely is the entire prompt + answer sequence?"

instead of:

> "How likely is this answer given this prompt?"

**DPO cares about the second one.**

---

## 9. Now connect it back to the DPO equation

Remember:

$$ \log \frac{\pi_\theta(y|x)} {\pi_{\rm ref}(y|x)} $$

Using the log identity:

$$ \log\frac{a}{b} = \log a-\log b $$

we calculate:

$$ \boxed{ \log\pi_\theta(y|x) - \log\pi_{\rm ref}(y|x) } $$

So for the chosen answer:

$$ \Delta_w= \log\pi_\theta(y_w|x) - \log\pi_{\rm ref}(y_w|x) $$

For rejected:

$$ \Delta_l= \log\pi_\theta(y_l|x) - \log\pi_{\rm ref}(y_l|x) $$

Then DPO uses:

$$ \boxed{ \Delta_w-\Delta_l } $$

and feeds it into the sigmoid.

---

## 10. The complete data flow

This is what we eventually implement:

```
             prompt
                │
       ┌────────┴────────┐
       ↓                 ↓
   chosen              rejected
       │                 │
       ↓                 ↓
     πθ model          πθ model
       │                 │
       ↓                 ↓
 log P(chosen)      log P(rejected)
       │                 │
       └────────┬────────┘
                │
         compare with
                │
       frozen πref model
                │
                ↓
     relative log probabilities
                │
                ↓
          DPO objective
                │
                ↓
          update πθ
```

---

### The key thing to lock in

When you see: $$ \pi_\theta(y|x) $$

don't imagine some mysterious single probability coming out of the Transformer.

Think:

$$ \boxed{ \pi_\theta(y|x) = \prod_t P_\theta(y_t|x,y_{<t}) } $$

and in practice:

$$ \boxed{ \log\pi_\theta(y|x) = \sum_t\log P_\theta(y_t|x,y_{<t}) } $$

**That is the bridge between the DPO mathematics and actual Transformer tensors.**

## Important optimization detail

For a given already-tokenized sequence, getting all the logits is **one forward pass**, not one forward pass per token.

### Example

**Input:** "The cat sat"

The Transformer processes the whole sequence in parallel:

```
"The"   "cat"   "sat"
  ↓       ↓       ↓
logits₁ logits₂ logits₃
```

Each position's logits predict the next token:

- "The"        → predicts "cat"
- "The cat"    → predicts "sat"
- "The cat sat"→ predicts next token

So for DPO, since the response already exists, we do:

$$ \boxed{\text{one forward pass} \rightarrow \text{logits for every position}} $$

Then we simply pick out the probability of the actual response token at each position.

# DPO From Scratch: Actual Working

Now we finally put the pieces together. Still no code yet — first understand exactly what one DPO training step does.

We'll use a tiny example all the way through.

---

## 1. Our training example

**Prompt:** "Explain gravity simply."

**Human gives:**

- **Chosen** = "Gravity pulls objects toward each other."
- **Rejected** = "Gravity is when things fall."

We have two models:

- $\pi_\theta$ = trainable LLM
- $\pi_{\rm ref}$ = frozen copy of SFT LLM

---

## 2. Both models score both answers

We feed the same prompt + chosen answer into both models.

And separately: prompt + rejected answer.

Each model gives us token logits.

We convert logits → log probabilities and sum the response-token log probabilities.

Suppose we get:

|  | Chosen | Rejected |
|--|--------|----------|
| $\pi_\theta$ | -8 | -12 |
| $\pi_{\rm ref}$ | -10 | -11 |

These are log probabilities of the entire response.

**Remember:** less negative = higher probability.

So our current model thinks:

$$ -8 > -12 $$

meaning it likes the chosen answer more.

---

## 3. Calculate the policy's "advantage over reference"

**For the chosen answer:**

$$ \Delta_w = \log\pi_\theta(y_w|x) - \log\pi_{\rm ref}(y_w|x) $$

Plug in: $$ \Delta_w=-8-(-10)=2 $$

Meaning:

> Relative to the reference model, our current model increased the chosen answer's log-probability by 2.

**For rejected:**

$$ \Delta_l = -12-(-11) = -1 $$

Meaning:

> Relative to the reference, our model decreased the rejected answer's log-probability by 1.

---

## 4. Compare those two changes

DPO cares about: $$ \Delta_w-\Delta_l $$

So: $$ 2-(-1)=3 $$

This is our **relative preference margin**.

**Big positive number = good.**

Why?

Because:

- **Chosen:** reference → current: -10 → -8 ↑
- **Rejected:** reference → current: -11 → -12 ↓

Exactly the direction we want.

---

## 5. Add β

DPO scales the margin by $\beta$.

Suppose: $$ \beta=0.1 $$

Then:

$$ z=\beta(\Delta_w-\Delta_l) = 0.1(3)=0.3 $$

---

## 6. Turn it into preference probability

Bradley-Terry gives us: $$ P(y_w\succ y_l) = \sigma(z) $$

So: $$ P=\sigma(0.3) \approx0.574 $$

Our model currently predicts about **57.4%** probability that the chosen answer should win.

---

## 7. Calculate the loss

We want this probability to approach 1.

So: $$ L=-\log(P) = -\log(0.574) \approx0.555 $$

**That's the loss for this training example.**

---

## 8. Then comes backpropagation

Now the important part.

We have: $$ L_{\rm DPO} $$

The gradient flows through: $$ \pi_\theta $$

and updates its weights.

But:

$$ \boxed{\pi_{\rm ref}\text{ receives NO gradient}} $$

It stays frozen.

So:

```
πθ
 ↓
DPO loss
 ↓
backprop
 ↓
update weights
```

while:

```
πref
 ↓
just provides reference log-probs
 ↓
NO update
```

---

## 9. What is the model actually learning?

This is the part I want you to really understand.

**DPO isn't simply saying:** "Make chosen probability bigger."

**It's saying:** "Change your probabilities so that the chosen answer becomes more favored relative to the rejected answer, while using the reference model as the baseline."

So ideally:

$$ \Delta_w\uparrow $$

and/or

$$ \Delta_l\downarrow $$

Therefore: $$ \Delta_w-\Delta_l\uparrow $$

Therefore: $$ \sigma(\cdot)\uparrow $$

Therefore: $$ L\downarrow $$

**That's the entire optimization loop.**

---

## 10. One training step

So one DPO example goes:

```
prompt + chosen
       ↓
      πθ ─────→ log Pθ(chosen)
       ↓
     πref ────→ log Pref(chosen)

prompt + rejected
       ↓
      πθ ─────→ log Pθ(rejected)
       ↓
     πref ────→ log Pref(rejected)

              ↓
       relative log-ratios
              ↓
       chosen - rejected
              ↓
              × β
              ↓
            sigmoid
              ↓
             -log
              ↓
          DPO loss
              ↓
        backprop into πθ
```

And then we repeat this over batches.

---

## 11. One subtle but VERY important thing

You might now think:

> "If DPO wants chosen probability up and rejected probability down, why not just use cross-entropy on chosen and be done?"

**Because that would be SFT.**

DPO uses the *relationship* between the two responses.

**Imagine:**

- Reference:
  - chosen   = 0.30
  - rejected = 0.20
- Current:
  - chosen   = 0.31
  - rejected = 0.01

The current model has dramatically separated them.

**DPO cares about that relative separation.**

That's the preference-learning signal that SFT doesn't directly have.

---

## One correction to keep in mind

Earlier I said DPO is "making the chosen relatively more likely than rejected."

That's correct, but there's an even better mental model:

**DPO is learning a preference boundary using the model's change from the reference as the implicit reward.**

The "reward" is never explicitly stored as:

```
reward = 3.72
```

Instead, it is implicitly represented by:

$$ \boxed{ \beta\log \frac{\pi_\theta(y|x)} {\pi_{\rm ref}(y|x)} } $$

**That is the core DPO idea.**

## But here's the important part

You might be thinking:

> "Wait. If our current model already gives chosen -8 and rejected -12, why does it still need training?"

**Because the margin isn't necessarily large enough.**

DPO wants the chosen response to win by an increasingly strong relative margin.

The optimizer changes the model's weights so that future forward passes produce a larger:

$$ \Delta_w-\Delta_l $$

---

## What gets changed inside the LLM?

This is normal neural-network training.

The DPO loss produces gradients:

$$ \frac{\partial L}{\partial\theta} $$

Then an optimizer such as Adam updates:

$$ \theta \leftarrow \theta-\eta\nabla_\theta L $$

So there's nothing magical happening to the weights.

**It's standard gradient descent.**

The only unusual part is how we constructed the loss.

---

## One subtle point: DPO isn't directly saying "make probability = X"

Suppose the chosen answer currently has: $$ P=0.2 $$

DPO doesn't say:

> "Make it 0.8."

Instead it says:

> "Increase the chosen-vs-rejected preference margin relative to the reference."

**That's why DPO is fundamentally a relative preference optimization method.**

---

## Complete picture

```
        preference dataset
               │
        ┌──────┴──────┐
        ↓             ↓
     chosen        rejected
        │             │
        └──────┬──────┘
               ↓
       ┌───────────────┐
       │               │
      πθ             πref
   trainable         frozen
       │               │
       ↓               ↓
   log Pθ            log Pref
       │               │
       └───────┬───────┘
               ↓
       relative log-ratios
               ↓
        chosen - rejected
               ↓
              β
               ↓
            sigmoid
               ↓
             loss
               ↓
          backprop
               ↓
             πθ ↑
```

**That's one DPO optimization step.**